# PPO Policy XAI Analysis (Inference-Only)

This notebook explains how the trained CNN policy behaves on one episode:
- episode trace and policy decisions,
- visual storyboard and topology animation,
- per-channel saliency maps (Gradient x Input),
- channel ablation impact,
- symmetry sanity checks.

No training is run here. CPU is enough. GPU is optional for faster inference.

In [ ]:
from pathlib import Path
from google.colab import drive
import os

drive.mount('/content/drive')
PROJECT_ROOT = Path('/content/drive/MyDrive/rl-quantum-circuit-routing')
%cd {PROJECT_ROOT}

RUN_NAME = 'ppo_2stages_gridv5_20260325_195948_s42'  # change if needed
TOPOLOGY = 'grid_3x3'  # try also 'linear_5' or 'heavy_hex_19'
CIRCUIT_DEPTH = 12
SEED = 42

MODEL_PATH = PROJECT_ROOT / 'runs' / RUN_NAME / 'best_model.pt'
print('PROJECT_ROOT:', PROJECT_ROOT)
print('RUN_NAME:', RUN_NAME)
print('MODEL_PATH exists:', MODEL_PATH.exists())

In [ ]:
import importlib
import subprocess, sys

def ensure_pkg(pkg: str, spec: str | None = None):
    try:
        importlib.import_module(pkg)
        return
    except Exception:
        target = spec if spec is not None else pkg
        print(f'Installing {target} ...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', target])

ensure_pkg('qiskit', 'qiskit==1.4.2')
ensure_pkg('networkx')
ensure_pkg('matplotlib')
ensure_pkg('numpy')
ensure_pkg('torch')

import torch, numpy as np, matplotlib.pyplot as plt, networkx as nx
from IPython.display import HTML
print('torch:', torch.__version__, 'cuda:', torch.cuda.is_available())

In [ ]:
from src.agent import SymmetricCNNActorCritic
from src.environment import QubitRoutingEnv
from src.circuit_utils import get_sabre_swap_count

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

def build_env(topology: str, depth: int, seed: int = 42):
    env = QubitRoutingEnv(
        topologies=[topology],
        matrix_size=27,
        circuit_depth=depth,
        max_steps=500,
        gamma_decay=0.5,
        distance_reward_coeff=0.015,
        completion_bonus=15.0,
        timeout_penalty=-8.0,
        gate_reward_coeff=1.0,
        step_penalty=-0.05,
        reverse_swap_penalty=-0.2,
        repeat_swap_penalty_coeff=-0.2,
        repeat_swap_penalty_cap=-2.0,
        no_progress_penalty_coeff=-0.03,
        no_progress_penalty_cap=-1.5,
        no_progress_terminate_streak=30,
        max_steps_per_two_qubit_gate=10.0,
        max_steps_min=60,
        max_steps_max=450,
        min_two_qubit_gates=8,
        circuit_generation_attempts=16,
        initial_mapping_strategy='sabre',
        seed=seed,
    )
    return env

def load_model(model_path: Path, matrix_size: int = 27, device: str = 'cpu'):
    model = SymmetricCNNActorCritic(matrix_size=matrix_size).to(device)
    state = torch.load(model_path, map_location=device)
    model.load_state_dict(state)
    model.eval()
    return model

def _edge_index_and_mask(env):
    topo = env._current_topo
    max_actions = int(env.action_space.n)
    edge_index = np.zeros((max_actions, 2), dtype=np.int64)
    edge_index[:topo['num_edges']] = np.asarray(topo['edges'], dtype=np.int64)
    action_mask = env.get_action_mask().astype(bool)
    return edge_index, action_mask

def run_greedy_trace(model, env, seed=42, max_steps=500):
    obs, info = env.reset(seed=seed)
    rows = []
    done = False
    truncated = False
    total_reward = 0.0
    prev_exec = int(info.get('total_gates_executed', 0))
    last_action = None

    while not done and not truncated and len(rows) < max_steps:
        edge_index, action_mask = _edge_index_and_mask(env)

        obs_t = torch.as_tensor(obs, dtype=torch.float32, device=DEVICE).unsqueeze(0)
        edge_t = torch.as_tensor(edge_index, dtype=torch.long, device=DEVICE).unsqueeze(0)
        mask_t = torch.as_tensor(action_mask, dtype=torch.bool, device=DEVICE).unsqueeze(0)
        with torch.no_grad():
            dist, _ = model.get_action_distribution(obs_t, edge_t, mask_t)
            logits = dist.logits.squeeze(0)
            probs = torch.softmax(logits, dim=-1)
            action = int(torch.argmax(logits).item())
            action_prob = float(probs[action].item())
            action_logit = float(logits[action].item())

        topo = env._current_topo
        edge_i, edge_j = topo['edges'][action]
        front_before = float(env._compute_front_layer_distance())

        next_obs, reward, done, truncated, info = env.step(action)
        front_after = float(env._compute_front_layer_distance())
        delta_dist = front_before - front_after

        step_exec = int(info.get('total_gates_executed', 0)) - prev_exec
        prev_exec = int(info.get('total_gates_executed', 0))
        was_backtrack = int(last_action is not None and action == last_action)
        last_action = action

        row = {
            'step': int(info.get('step_count', len(rows) + 1)),
            'obs': obs.copy(),
            'edge_index': edge_index.copy(),
            'action_mask': action_mask.copy(),
            'action_index': action,
            'edge_i': int(edge_i),
            'edge_j': int(edge_j),
            'action_prob': action_prob,
            'action_logit': action_logit,
            'reward': float(reward),
            'step_gates_executed': int(step_exec),
            'total_gates_executed': int(info.get('total_gates_executed', 0)),
            'remaining_gates': int(info.get('remaining_gates', 0)),
            'front_dist_before': front_before,
            'front_dist_after': front_after,
            'delta_dist': float(delta_dist),
            'was_immediate_backtrack': was_backtrack,
            'done': int(done),
            'truncated': int(truncated),
        }
        rows.append(row)
        total_reward += float(reward)
        obs = next_obs

    sabre_swaps = int(get_sabre_swap_count(env.circuit, env._current_topo['coupling_map']))
    ppo_swaps = int(info.get('total_swaps', 0))
    improve = 100.0 * (sabre_swaps - ppo_swaps) / max(1, sabre_swaps)

    summary = {
        'topology': str(env._current_topo['name']),
        'num_qubits': int(env._current_topo['n_physical']),
        'all_edges': [list(e) for e in env._current_topo['edges']],
        'steps': len(rows),
        'done': bool(done),
        'truncated': bool(truncated),
        'total_reward': float(total_reward),
        'ppo_swaps': int(ppo_swaps),
        'sabre_swaps': int(sabre_swaps),
        'improvement_pct': float(improve),
    }
    return rows, summary

env = build_env(TOPOLOGY, CIRCUIT_DEPTH, SEED)
model = load_model(MODEL_PATH, matrix_size=env.N, device=DEVICE)
rows, summary = run_greedy_trace(model, env, seed=SEED)

print('device:', DEVICE)
print('summary:', summary)
print('first row keys:', list(rows[0].keys()) if rows else 'no rows')

In [ ]:
steps = np.array([r['step'] for r in rows], dtype=np.int32)
rewards = np.array([r['reward'] for r in rows], dtype=np.float32)
probs = np.array([r['action_prob'] for r in rows], dtype=np.float32)
remaining = np.array([r['remaining_gates'] for r in rows], dtype=np.int32)
exec_step = np.array([r['step_gates_executed'] for r in rows], dtype=np.int32)
exec_cum = np.array([r['total_gates_executed'] for r in rows], dtype=np.int32)
delta_dist = np.array([r['delta_dist'] for r in rows], dtype=np.float32)
backtrack = np.array([r['was_immediate_backtrack'] for r in rows], dtype=np.int32)
action_idx = np.array([r['action_index'] for r in rows], dtype=np.int32)
edge_pairs = [tuple(sorted((r['edge_i'], r['edge_j']))) for r in rows]

from collections import Counter
edge_counts = Counter(edge_pairs)
all_edges = [tuple(sorted(e)) for e in summary['all_edges']]
topology = summary['topology']
nq = summary['num_qubits']

if topology == 'grid_3x3' and nq == 9:
    pos = {i: (i % 3, -(i // 3)) for i in range(nq)}
elif topology.startswith('linear'):
    pos = {i: (i, 0.0) for i in range(nq)}
else:
    gtmp = nx.Graph()
    gtmp.add_nodes_from(range(nq))
    gtmp.add_edges_from(all_edges)
    pos = nx.spring_layout(gtmp, seed=7)

fig, axs = plt.subplots(2, 2, figsize=(14, 9))

ax = axs[0, 0]
ax.plot(steps, rewards, label='reward', color='#1f77b4')
ax.plot(steps, delta_dist, label='delta_dist', color='#2ca02c', alpha=0.85)
ax2 = ax.twinx()
ax2.plot(steps, probs, label='action_prob', color='#d62728', alpha=0.65)
ax.set_title('Decision Signal Over Time')
ax.set_xlabel('step')
ax.set_ylabel('reward / delta_dist')
ax2.set_ylabel('action_prob')
ax.grid(alpha=0.25)

ax = axs[0, 1]
ax.plot(steps, remaining, label='remaining_gates', color='#9467bd')
ax.plot(steps, exec_cum, label='executed_gates_cum', color='#ff7f0e')
ax.bar(steps, exec_step, alpha=0.25, label='executed_this_step', color='gray')
ax.set_title('Routing Progress')
ax.set_xlabel('step')
ax.grid(alpha=0.25)
ax.legend()

ax = axs[1, 0]
colors = np.where(backtrack == 1, '#d62728', '#1f77b4')
sizes = 30 + 140 * probs
ax.scatter(steps, action_idx, c=colors, s=sizes, alpha=0.7)
ax.set_title('Action Index (red = immediate backtrack)')
ax.set_xlabel('step')
ax.set_ylabel('action_index')
ax.grid(alpha=0.25)

ax = axs[1, 1]
G = nx.Graph()
G.add_nodes_from(range(nq))
G.add_edges_from(all_edges)
nx.draw_networkx_nodes(G, pos, node_size=420, node_color='#f2f2f2', edgecolors='black', ax=ax)
nx.draw_networkx_labels(G, pos, font_size=8, ax=ax)
base_edges = [e for e in all_edges if edge_counts.get(e, 0) == 0]
used_edges = [e for e in all_edges if edge_counts.get(e, 0) > 0]
nx.draw_networkx_edges(G, pos, edgelist=base_edges, width=1.0, edge_color='#cccccc', ax=ax)
if used_edges:
    mx = max(edge_counts[e] for e in used_edges)
    widths = [1.5 + 6.0 * edge_counts[e] / mx for e in used_edges]
    nx.draw_networkx_edges(G, pos, edgelist=used_edges, width=widths, edge_color='#d62728', ax=ax)
ax.set_title('Topology Edge Usage (thicker = used more)')
ax.axis('off')

fig.suptitle(
    f"Policy Storyboard | topo={topology} | ppo_swaps={summary['ppo_swaps']} | sabre_swaps={summary['sabre_swaps']} | improve={summary['improvement_pct']:.2f}%",
    fontsize=12
)
plt.tight_layout()
plt.show()

In [ ]:
from matplotlib.animation import FuncAnimation

fig, ax = plt.subplots(figsize=(6, 5))
G2 = nx.Graph()
G2.add_nodes_from(range(nq))
G2.add_edges_from(all_edges)

def draw_frame(k):
    ax.clear()
    nx.draw_networkx_nodes(G2, pos, node_size=420, node_color='#fafafa', edgecolors='black', ax=ax)
    nx.draw_networkx_labels(G2, pos, font_size=8, ax=ax)
    nx.draw_networkx_edges(G2, pos, edgelist=all_edges, width=1.0, edge_color='#d0d0d0', ax=ax)

    e = edge_pairs[k]
    nx.draw_networkx_edges(G2, pos, edgelist=[e], width=5.0, edge_color='#d62728', ax=ax)

    ax.set_title(
        f"step={steps[k]} edge={e} reward={rewards[k]:+.2f} rem={remaining[k]} p={probs[k]:.2f}"
    )
    ax.axis('off')

anim = FuncAnimation(fig, draw_frame, frames=len(rows), interval=250, repeat=False)
plt.close(fig)
HTML(anim.to_jshtml())

In [ ]:
# XAI on one selected step: saliency (Gradient x Input) + channel ablation

if len(rows) == 0:
    raise RuntimeError('No trace rows to analyze.')

candidate = [i for i, r in enumerate(rows) if r['remaining_gates'] > 0]
STEP_IDX = candidate[min(10, len(candidate) - 1)] if candidate else min(5, len(rows) - 1)
r = rows[STEP_IDX]

obs = r['obs']
edge_index = r['edge_index']
action_mask = r['action_mask']
action = int(r['action_index'])

obs_t = torch.as_tensor(obs, dtype=torch.float32, device=DEVICE).unsqueeze(0)
obs_t.requires_grad_(True)
edge_t = torch.as_tensor(edge_index, dtype=torch.long, device=DEVICE).unsqueeze(0)
mask_t = torch.as_tensor(action_mask, dtype=torch.bool, device=DEVICE).unsqueeze(0)

dist, _ = model.get_action_distribution(obs_t, edge_t, mask_t)
logits = dist.logits
target_logit = logits[0, action]
target_logit.backward()

grad = obs_t.grad.detach().cpu().numpy()[0]
obs_np = obs_t.detach().cpu().numpy()[0]
sal = np.abs(grad * obs_np)

sal_norm = sal.copy()
for c in range(3):
    m = sal_norm[c].max()
    if m > 1e-12:
        sal_norm[c] /= m

base_probs = torch.softmax(logits.detach(), dim=-1)[0]
base_prob = float(base_probs[action].item())
base_logit = float(logits[0, action].detach().item())

ablation = []
for c in range(3):
    obs_abl = obs.copy()
    obs_abl[c] = 0.0
    obs_abl_t = torch.as_tensor(obs_abl, dtype=torch.float32, device=DEVICE).unsqueeze(0)
    with torch.no_grad():
        dist2, _ = model.get_action_distribution(obs_abl_t, edge_t, mask_t)
        logits2 = dist2.logits[0]
        probs2 = torch.softmax(logits2, dim=-1)
    ablation.append({
        'channel': c,
        'delta_logit': float(logits2[action].item() - base_logit),
        'delta_prob': float(probs2[action].item() - base_prob),
    })

titles = ['Channel0 Topology', 'Channel1 Mapping', 'Channel2 Demand']
fig, axs = plt.subplots(1, 3, figsize=(15, 4.5))
for c in range(3):
    im = axs[c].imshow(sal_norm[c], cmap='magma')
    axs[c].set_title(titles[c])
    axs[c].set_xticks([])
    axs[c].set_yticks([])
    plt.colorbar(im, ax=axs[c], fraction=0.046, pad=0.04)
fig.suptitle(
    f"Saliency (Grad x Input) for chosen action at step={r['step']} edge=({r['edge_i']},{r['edge_j']})",
    fontsize=12
)
plt.tight_layout()
plt.show()

print('base action:', action, 'base_prob:', round(base_prob, 4), 'base_logit:', round(base_logit, 4))
for a in ablation:
    print(a)

In [ ]:
# Symmetry sanity checks on one observation
obs = rows[min(5, len(rows)-1)]['obs']
obs_t = torch.as_tensor(obs, dtype=torch.float32, device=DEVICE).unsqueeze(0)

with torch.no_grad():
    score, _ = model.forward(obs_t)
score_np = score[0].detach().cpu().numpy()
sym_err = np.max(np.abs(score_np - score_np.T))

obs_T = np.transpose(obs, (0, 2, 1))
obs_T_t = torch.as_tensor(obs_T, dtype=torch.float32, device=DEVICE).unsqueeze(0)
with torch.no_grad():
    score_T, _ = model.forward(obs_T_t)
score_T_np = score_T[0].detach().cpu().numpy()
equiv_err = np.mean(np.abs(score_np - score_T_np.T))

print('Symmetry error max |S - S^T|:', float(sym_err))
print('Transpose-equivariance proxy mean |S(X) - S(X^T)^T|:', float(equiv_err))

## How to read these results
- If `delta_prob` after channel ablation is strongly negative, that channel is important for this decision.
- If saliency mostly lights up in Demand (channel 2), the policy is using gate-pressure signal.
- If action index plot has long red bands, behavior is loop-prone (immediate backtracking).
- Lower swap ratio toward 1.0 and lower timeout trend are good signs against SABRE.